In [ ]:
!pip install datasets sentencepiece sacrebleu rouge-score torch

In [ ]:
from datasets import load_dataset
ds=load_dataset("uqa/UQA")
print(ds)

ex=ds["train"][0]
print(ex.keys())

print(ex["question"])
print(ex["answer"])

n_total=len(ds["train"])
n_ans=sum(not row["is_impossible"] and bool(row["answer"]) for row in ds["train"])
print(f"train rows: {n_total}, answerable: {n_ans}")

In [ ]:
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/GenAI-Dataset (1)')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

ANS_OPEN = "<ans>"
ANS_CLOSE = "</ans>"
SENT_DELIMS = "\u06D4\u061F!" 

def split_sentences(text):
  start = 0
  for i, ch in enumerate(text):
    if ch in SENT_DELIMS:
      yield start, i + 1, text[start : i + 1]
      start = i + 1
  if start < len(text):
    yield start, len(text), text[start:]

def make_pair(example, max_src=60, max_tgt=25):
  answer = example["answer"]
  # Both checks are explicit so the answerability rule is easy to verify.
  if example["is_impossible"] or not answer:
    return None
  a_start = example["answer_start"]
  a_text = answer
  context = example["context"]
  for s, e, sent in split_sentences(context):
    # Check if answer starts inside this sentence
    if s <= a_start < e:
      rel = a_start - s  # Offset inside this specific sentence
      # Integrity check: does the text at this slice match the answer text?
      if sent[rel : rel + len(a_text)] != a_text:
        return None  # Offset mismatch -> skip
      # Insert the <ans> tags
      src = (
          sent[:rel]
          + " "
          + ANS_OPEN
          + " "
          + a_text
          + " "
          + ANS_CLOSE
          + " "
          + sent[rel + len(a_text) :]
      ).strip()
      # Normalise extra whitespaces
      src = " ".join(src.split())
      tgt = " ".join(example["question"].split())
      # Apply length filters (whitespace tokens)
      src_len = len(src.split())
      tgt_len = len(tgt.split())
      if src_len > max_src or tgt_len > max_tgt:
        return None
      return src, tgt
  return None


In [ ]:
def build_split(split, out_path):
  pairs = [p for p in map(make_pair, split) if p is not None]
  pairs = list(dict.fromkeys(pairs))  # Remove exact duplicates, keep original order.
  with open(out_path, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(
        f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\"
    )
    w.writerows(pairs)
  print(f"{out_path}: {len(pairs)} pairs")
  return pairs
# Build train and valid
train_pairs = build_split(ds["train"], DRIVE_DIR / "train.tsv")
valid_pairs = build_split(ds["validation"], DRIVE_DIR / "valid.tsv")


wiki_ds = load_dataset("uqa/Wiki-UQA")

wiki_split = wiki_ds["train"]
wiki_pairs = build_split(wiki_split, DRIVE_DIR / "wiki_test.tsv")

#inspecting the first 3 pairs
for i in range(3):
  src, tgt = train_pairs[i]
  print(f"{i+1} : ")
  print("Source (Sentence + <ans>):", src)
  print("Target (Question):         ", tgt)

In [ ]:
# Plot length histograms for Train and Validation pairs
def plot_histograms(pairs, split_name="Train"):
    src_lens = [len(src.split()) for src, tgt in pairs]
    tgt_lens = [len(tgt.split()) for src, tgt in pairs]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Source length histogram
    axes[0].hist(src_lens, bins=30, color="steelblue", edgecolor="black")
    axes[0].set_title(f"{split_name} - Source Lengths (words)")
    axes[0].set_xlabel("Token Count (whitespace words)")
    axes[0].set_ylabel("Frequency")
    axes[0].axvline(np.mean(src_lens), color="red", linestyle="--", label=f"Mean: {np.mean(src_lens):.1f}")
    axes[0].legend()

    # Target length histogram
    axes[1].hist(tgt_lens, bins=25, color="coral", edgecolor="black")
    axes[1].set_title(f"{split_name} - Target Lengths (words)")
    axes[1].set_xlabel("Token Count (whitespace words)")
    axes[1].set_ylabel("Frequency")
    axes[1].axvline(np.mean(tgt_lens), color="blue", linestyle="--", label=f"Mean: {np.mean(tgt_lens):.1f}")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(DRIVE_DIR / f"{split_name.lower()}_length_histogram.png", dpi=160)
    plt.show()

plot_histograms(train_pairs, "Train Split")
plot_histograms(valid_pairs, "Validation Split")